In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_bounds
import geopandas as gpd
from shapely.geometry import box

In [ ]:
# Bijeljina (WGS84 lon/lat)
BIJELJINA_BOUNDS = [19.18, 44.73, 19.27, 44.81]  # lon_min, lat_min, lon_max, lat_max

T1_PATH = 'data/bijeljina_t1.png'
T2_PATH = 'data/bijeljina_t2.png'
OSM_PBF = 'data/bih.osm.pbf'

In [ ]:
def load_sentinel_pair(t1_path, t2_path):
    """Loading PNG with no resizing"""
    img1 = cv2.cvtColor(cv2.imread(t1_path), cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    img2 = cv2.cvtColor(cv2.imread(t2_path), cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    
    global IMG_HEIGHT, IMG_WIDTH
    IMG_HEIGHT, IMG_WIDTH = img1.shape[:2]
    print(f"PNG shape: {IMG_HEIGHT}x{IMG_WIDTH}")
    
    return img1, img2

img1, img2 = load_sentinel_pair(T1_PATH, T2_PATH)
print(f"Sentinel shape: {img1.shape}")

In [ ]:
def rasterize_osm(pbf_path, bounds, height=IMG_HEIGHT, width=IMG_WIDTH):
    """OSM zgrade: raster maska"""
    gdf = gpd.read_file(pbf_path, layer='multipolygons')
    gdf_buildings = gdf[gdf['building'].notna()].copy()
    bounds_poly = box(*bounds)
    gdf_bijeljina = gdf_buildings[gdf_buildings.intersects(bounds_poly)]
    
    # same as png shape
    transform = from_bounds(*bounds, width, height)
    
    shapes = [(row.geometry, 1) for _, row in gdf_bijeljina.iterrows()]
    osm_mask = rasterize(shapes, out_shape=(height, width),
                        transform=transform, dtype=np.uint8, all_touched=True)

    osm_mask = osm_mask.astype(np.float32) / 255.0
    return osm_mask

osm_mask = rasterize_osm(OSM_PBF, BIJELJINA_BOUNDS)
print(f"OSM shape: {osm_mask.shape} == Sentinel: {img1.shape[:2]}")

In [ ]:
def compute_change_diff(img1, img2):
    """Pseudo-mask of differences (95 threshold)"""
    diff_rgb = np.mean(np.abs(img2 - img1), axis=2)
    change_thresh = np.percentile(diff_rgb, 95)
    change_mask = (diff_rgb > change_thresh).astype(np.uint8)
    return change_mask

change_mask = compute_change_diff(img1, img2)
print(f"Change pixels: {np.sum(change_mask)} ({np.sum(change_mask) / (IMG_HEIGHT * IMG_WIDTH) * 100:.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8), dpi=150)

# 1. Sentinel 2020
axes[0].imshow(img2)
axes[0].set_title('Sentinel-2 Bijeljina 2020', fontsize=16, fontweight='bold')
axes[0].axis('off')

# 2. OVERLAY i change maska
axes[1].imshow(img2)
axes[1].imshow(change_mask, cmap='Reds', alpha=0.5)
axes[1].set_title(f'Sentinel + Change maska\n{np.sum(change_mask)} piksela', fontsize=16, fontweight='bold')
axes[1].axis('off')

plt.suptitle('Gdje su promjene na satelitu?', fontsize=20, fontweight='bold')
plt.tight_layout()
plt.savefig('reports/bijeljina_overlay.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# oko centra grada 3x3km
CENTRAL_BIJELJINA = [19.21, 44.75, 19.24, 44.78]  # lon_min, lat_min, lon_max, lat_max

In [ ]:
def get_central_osm_final(pbf_path, bounds=CENTRAL_BIJELJINA):
    gdf = gpd.read_file(pbf_path, layer='multipolygons')
    gdf_buildings = gdf[gdf['building'].notna()].copy()
    
    bounds_poly = box(*bounds)
    gdf_central = gdf_buildings[gdf_buildings.intersects(bounds_poly)]
    
    # industrial flag (columns: building, man-made)
    gdf_central['industrial'] = (
        (gdf_central['building'] == 'industrial') |
        (gdf_central['building'].str.contains('factory|works', case=False, na=False)) |
        (gdf_central['man_made'] == 'works')
    ).astype(int)
    
    print(f"OSM: {len(gdf_central)} zgrada")
    print(f"Industrial: {gdf_central['industrial'].sum()}")
    return gdf_central

osm_central = get_central_osm_final('data/bih.osm.pbf')


In [ ]:
# Central crop 512x512
crop_size = 512
center_h, center_w = IMG_HEIGHT//2, IMG_WIDTH//2
h_start = max(0, center_h - crop_size//2)
w_start = max(0, center_w - crop_size//2)

img1_crop = img1[h_start:h_start+crop_size, w_start:w_start+crop_size]
img2_crop = img2[h_start:h_start+crop_size, w_start:w_start+crop_size]

# Central OSM + change mask
osm_mask_crop = rasterize_osm('data/bih.osm.pbf', CENTRAL_BIJELJINA, crop_size, crop_size)
change_crop = compute_change_diff(img1_crop, img2_crop)

print(f"Crop: {crop_size}x{crop_size}")
print(f"OSM pixels: {np.sum(osm_mask_crop > 0)}")
print(f"Change pixels: {np.sum(change_crop)}")


In [ ]:
import shapely

def raster_to_geojson(raster_mask, transform):
    """Debug version"""
    mask_binary = (raster_mask > 0.5).astype(np.uint8)
    print(f"Mask binary piksela: {np.sum(mask_binary)}")
    
    results = list(shapes(mask_binary, transform=transform))
    print(f"Shapes found: {len(results)}")
    
    geoms = []
    for geom_shape, value in results:
        geom = shapely.geometry.shape(geom_shape)
        if geom.is_valid and geom.area > 0.000005:  # ~0.5m²
            geoms.append(geom)
    
    gdf = gpd.GeoDataFrame({'geometry': geoms}, crs='EPSG:4326')
    if len(gdf) == 0:
        print("No valid geometries!")
    
    gdf['area_m2'] = gdf.geometry.area * 111320**2
    print(f"Vraćeno {len(gdf)} vektora, prosječna površina: {gdf['area_m2'].mean():.0f}m²")
    return gdf


transform_crop = from_bounds(*CENTRAL_BIJELJINA, crop_size, crop_size)
change_vectors_crop = raster_to_geojson(change_crop, transform_crop)

In [ ]:
print("=== CENTRALNI DEBUG ===")
print(f"change_crop shape: {change_crop.shape}")
print(f"change piksela: {np.sum(change_crop)} / {change_crop.size} = {np.sum(change_crop)/change_crop.size*100:.2f}%")

# Proveri ima li change piksela
if np.sum(change_crop) == 0:
    print("NO CHANGE pixels— threshold too high!")
    # Sniži threshold
    diff_rgb = np.mean(np.abs(img2_crop - img1_crop), axis=2)
    change_thresh = np.percentile(diff_rgb, 90)  # 90% umesto 95%
    change_crop = (diff_rgb > change_thresh).astype(np.uint8)
    print(f"Calculated again, with 90% threshold: {np.sum(change_crop)} piksela")


In [ ]:
def central_flag_fixed(osm_gdf, change_vectors, buffer_dist=0.0004):
    if len(change_vectors) == 0:
        print("❌ Nema change vektora")
        return gpd.GeoDataFrame()
    
    flags = []
    for idx, cv in change_vectors.iterrows():
        if cv['area_m2'] < 20:
            continue
            
        buffer_zone = cv.geometry.buffer(buffer_dist)
        nearby = osm_gdf[osm_gdf.intersects(buffer_zone)]
        
        ind_count = nearby['industrial'].sum()
        total_bldgs = len(nearby)
        
        flags.append({
            'flag_type': 'industrial' if ind_count > 0 else 'residential' if total_bldgs > 0 else 'new_zone',
            'change_area_m2': cv['area_m2'],
            'industrial_nearby': int(ind_count),
            'total_nearby': total_bldgs,
            'geometry': cv.geometry
        })
    
    flags_df = gpd.GeoDataFrame(flags)
    print(f"✅ Flags kreirano: {len(flags_df)}")
    return flags_df

# Run
flags_central = central_flag_fixed(osm_central, change_vectors_crop)
print(flags_central['flag_type'].value_counts())


In [ ]:
import skimage
from skimage import measure,segmentation

In [ ]:
def get_central_osm_final(pbf_path, bounds=CENTRAL_BIJELJINA):
    """Finalna verzija — koristi stvarne kolone"""
    gdf = gpd.read_file(pbf_path, layer='multipolygons')
    gdf_buildings = gdf[gdf['building'].notna()].copy()
    
    bounds_poly = box(*bounds)
    gdf_central = gdf_buildings[gdf_buildings.intersects(bounds_poly)]
    
    # Industrial flag (iz kolone 'building' + 'man_made')
    gdf_central['industrial'] = (
        (gdf_central['building'] == 'industrial') |
        (gdf_central['building'].str.contains('factory|works', case=False, na=False)) |
        (gdf_central['man_made'] == 'works')
    ).astype(int)
    
    print(f"✅ Centralni OSM: {len(gdf_central)} zgrada")
    print(f"   Industrijskih: {gdf_central['industrial'].sum()}")
    return gdf_central

# Run
osm_central = get_central_osm_final('data/bih.osm.pbf')


In [ ]:
# 1. Centralni crop
crop_size = 512
center_h, center_w = IMG_HEIGHT//2, IMG_WIDTH//2
h_start, w_start = center_h-crop_size//2, center_w-crop_size//2
img2_crop = img2[h_start:h_start+crop_size, w_start:w_start+crop_size]

# 2. Centralne maske
osm_crop = rasterize_osm('data/bih.osm.pbf', CENTRAL_BIJELJINA, crop_size, crop_size)
change_crop = compute_change_diff(img1_crop, img2_crop)

# 3. FLAG LOGIKA (pixel‑based)
osm_change_overlap = (osm_crop > 0) & (change_crop > 0)
industrial_pixels = np.sum(osm_change_overlap)  # Zgrade u promeni

print(f"Zgrada piksela u promeni: {industrial_pixels}")
print(f"Ukupno change piksela: {np.sum(change_crop)}")


In [ ]:
# 1. Original PNG bounds (iz bijeljina_test2)
PNG_BOUNDS = [19.18, 44.73, 19.27, 44.81]  # lon_min, lat_min, lon_max, lat_max
PNG_HEIGHT, PNG_WIDTH = img2.shape[:2]

# 2. Transform PNG
png_transform = from_bounds(*PNG_BOUNDS, PNG_WIDTH, PNG_HEIGHT)

# 3. Geo centar
central_lon = (CENTRAL_BIJELJINA[0] + CENTRAL_BIJELJINA[2]) / 2  # 19.225
central_lat = (CENTRAL_BIJELJINA[1] + CENTRAL_BIJELJINA[3]) / 2  # 44.765

# 4. Pixel koordinate centralnog dijela slike
central_px_x, central_px_y = ~png_transform * (central_lon, central_lat)
print(f"Centralni geo: lon={central_lon:.4f}, lat={central_lat:.4f}")
print(f"Centralni pikseli: x={central_px_x:.1f}, y={central_px_y:.1f}")

# 5. crop oko centralnih piksela
crop_size = 512
h_start = max(0, int(central_px_y - crop_size/2))
w_start = max(0, int(central_px_x - crop_size/2))
h_end = min(PNG_HEIGHT, h_start + crop_size)
w_end = min(PNG_WIDTH, w_start + crop_size)

img1_crop = img1[h_start:h_end, w_start:w_end]
img2_crop = img2[h_start:h_end, w_start:w_end]
actual_crop_size = img2_crop.shape[0]

print(f"Centralni crop: {img2_crop.shape} (pikseli {h_start}:{h_end}, {w_start}:{w_end})")


In [ ]:
# Centralni OSM mask (isti crop_size)
osm_mask_crop = rasterize_osm('data/bih.osm.pbf', CENTRAL_BIJELJINA, 
                              actual_crop_size, actual_crop_size)

# Promjena u centru
change_crop = compute_change_diff(img1_crop, img2_crop)

print(f"osm_mask_crop: {osm_mask_crop.shape}")
print(f"change_crop: {change_crop.shape}")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

axes[0,0].imshow(img2_crop)
axes[0,0].set_title('Centralni PNG')
axes[0,0].axis('off')

axes[0,1].imshow(osm_mask_crop.squeeze(), cmap='Greens')
axes[0,1].set_title(f'Centralni OSM\n{np.sum(osm_mask_crop > 0)} piksela')
axes[0,1].axis('off')

axes[1,0].imshow(change_crop, cmap='Reds')
axes[1,0].set_title(f'Promjena u centru\n{np.sum(change_crop)} piksela')
axes[1,0].axis('off')


axes[1,1].imshow(img2_crop)
axes[1,1].imshow(osm_mask_crop.squeeze(), cmap='Greens', alpha=0.5)
axes[1,1].imshow(change_crop, cmap='Reds', alpha=0.6)
overlap = np.sum((osm_mask_crop > 0) & (change_crop))
axes[1,1].set_title(f'Promjena: {overlap} piksela')
axes[1,1].axis('off')

plt.tight_layout()
plt.savefig('central_geo_aligned.png', dpi=300)
plt.show()
